# Three-block DFM planning experiment

Train on **9 problem instances** and evaluate on **9 held-out instances**, all using blocks A, B, C. Each stack below is ordered **bottom to top**. Initial arrangements are disjoint across splits; goals and the domain vocabulary may overlap. No held-out target plans enter training.

Run all cells in order using the project virtual environment. The default CPU-friendly model trains for 50 epochs. This small experiment measures performance; it does not guarantee generalization.


In [1]:
from main import (
    ExperimentConfig, make_problem_splits, build_training_dataset,
    train_model, evaluate_model, save_experiment, print_evaluation,
)

config = ExperimentConfig()
train_instances, test_instances = make_problem_splits(config.seed)
for split, instances in (("TRAIN", train_instances), ("HELD OUT", test_instances)):
    print(split)
    for item in instances:
        print(f"  {item['id']}: {item['initial_stacks']} -> {item['goal_stacks']}")


TRAIN
  train_01: (('B',), ('C', 'A')) -> (('C', 'A', 'B'),)
  train_02: (('A', 'C', 'B'),) -> (('A', 'B', 'C'),)
  train_03: (('A',), ('C', 'B')) -> (('B', 'C', 'A'),)
  train_04: (('B', 'A', 'C'),) -> (('B', 'C', 'A'),)
  train_05: (('C', 'B', 'A'),) -> (('A', 'B'), ('C',))
  train_06: (('A', 'C'), ('B',)) -> (('B', 'A'), ('C',))
  train_07: (('B',), ('C', 'A')) -> (('A', 'B'), ('C',))
  train_08: (('A', 'C', 'B'),) -> (('A',), ('B', 'C'))
  train_09: (('A',), ('C', 'B')) -> (('A', 'B', 'C'),)
HELD OUT
  test_01: (('C', 'A', 'B'),) -> (('B',), ('C', 'A'))
  test_02: (('B', 'A'), ('C',)) -> (('A',), ('B', 'C'))
  test_03: (('A', 'B'), ('C',)) -> (('B',), ('C', 'A'))
  test_04: (('A', 'B', 'C'),) -> (('A', 'C', 'B'),)
  test_05: (('A',), ('B',), ('C',)) -> (('A', 'B'), ('C',))
  test_06: (('A',), ('B', 'C')) -> (('B', 'A'), ('C',))
  test_07: (('B', 'C', 'A'),) -> (('A',), ('B',), ('C',))
  test_08: (('C', 'A', 'B'),) -> (('A',), ('B', 'C'))
  test_09: (('B', 'A'), ('C',)) -> (('C', 'B

## Build the training dataset

Enumerate goal-satisfying plans up to 8 actions, keep at most 128 per instance using a fixed seed, and verify every retained target with the plan validator. A shared codec gives every problem the same vocabulary and tensor shapes. Only the training instances are passed to this step.


In [2]:
dataset, codec, counts = build_training_dataset(train_instances, config)
for row in counts:
    print(f"{row['id']}: {row['training_plans']} retained / {row['enumerated_plans']} valid plans")
print(f"Total training examples: {len(dataset)}")
print(f"Context shape: {tuple(dataset[0]['context'].shape)}")
print(f"Plan shape: {tuple(dataset[0]['plan'].shape)}")


train_01: 52 retained / 52 valid plans
train_02: 1 retained / 1 valid plans
train_03: 11 retained / 11 valid plans
train_04: 1 retained / 1 valid plans
train_05: 37 retained / 37 valid plans
train_06: 72 retained / 72 valid plans
train_07: 72 retained / 72 valid plans
train_08: 37 retained / 37 valid plans
train_09: 37 retained / 37 valid plans
Total training examples: 320
Context shape: (40, 3)
Plan shape: (9, 3)


## Train the conditional DFM

The existing `train_dfm` loop learns to reconstruct clean plans from partially masked plans, conditioned on the initial state and goal. `history` records loss and masked-token accuracy for each epoch. These are training metrics, not planning success rates.


In [3]:
model, history = train_model(dataset, codec, config)


/home/itay/Documents/DFMplanning/Planning_Discrete_flow_matching/model.py:263: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(
Epoch 001 | loss=1.7785 | masked accuracy=0.2266
Epoch 002 | loss=1.6421 | masked accuracy=0.2670
Epoch 003 | loss=1.5435 | masked accuracy=0.2944
Epoch 004 | loss=1.4551 | masked accuracy=0.3170
Epoch 005 | loss=1.3792 | masked accuracy=0.3285
Epoch 006 | loss=1.2921 | masked accuracy=0.3360
Epoch 007 | loss=1.2303 | masked accuracy=0.3525
Epoch 008 | loss=1.1326 | masked accuracy=0.3813
Epoch 009 | loss=1.0667 | masked accuracy=0.4068
Epoch 010 | loss=1.0123 | masked accuracy=0.4166
Epoch 011 | loss=0.9226 | masked accuracy=0.4414
Epoch 012 | loss=0.8743 | masked accuracy=0.4641
Epoch 013 | loss=0.8386 | masked accuracy=0.4558
Epoch 014 | loss=0.7655 | masked accuracy=0.4847
Epoch 015 | loss=0.7653 | masked accuracy=0.4829
Epoch 016 | loss=0.72

## Evaluate on the nine unseen problems

Generate 32 raw plans per problem with 64 DFM sampling steps. Strict validation checks formatting, action applicability, and final goal satisfaction. All samples count in the denominator, including malformed plans. There is no repair or reference-plan substitution.

- **Satisfaction rate:** goal-satisfying samples / all 288 generated samples.
- **Executable rate:** correctly formatted, executable samples / all samples (the goal may still be unmet).
- **Problems solved:** held-out problems with at least one satisfying sample / 9. This depends on the sampling budget and is not single-sample accuracy.


In [4]:
evaluation = evaluate_model(model, test_instances, codec, config)
print_evaluation(evaluation)
print(f"Executable rate: {evaluation['summary']['executable_rate']:.1%}")



Held-out evaluation (raw samples, strict plan validation)
Problem        Executable   Satisfying   Satisfaction
test_01           0/32         0/32             0.0%
test_02           0/32         0/32             0.0%
test_03           0/32         0/32             0.0%
test_04           0/32         0/32             0.0%
test_05           0/32         0/32             0.0%
test_06           0/32         0/32             0.0%
test_07           0/32         0/32             0.0%
test_08           0/32         0/32             0.0%
test_09           0/32         0/32             0.0%
Overall satisfaction: 0.0%; problems solved: 0/9
Executable rate: 0.0%


## Save and inspect results

Save the dataset, trained weights, exact problem split, training history, and every generated token tensor with its validation outcome. Failure examples help distinguish formatting issues from action-precondition or goal failures. Zero observed successes means this run did not produce a solution; it does not imply the planning problems are unsatisfiable.


In [5]:
save_experiment(
    config, train_instances, test_instances, dataset, codec, counts,
    model, history, evaluation,
)
print(f"Artifacts saved to {config.output_dir}")
for sample in [s for s in evaluation["samples"] if not s["solves_problem"]][:5]:
    print(f"{sample['id']} sample {sample['sample']}: {sample['reason']}")


Artifacts saved to artifacts/three_blocks
test_01 sample 0: Action pick_up has unexpected arguments: ['pick_up', 'C', 'A']
test_01 sample 1: PAD encountered before END at row 0.
test_01 sample 2: Action pick_up has unexpected arguments: ['pick_up', 'B', 'A']
test_01 sample 3: Action put_down has unexpected arguments: ['put_down', 'C', 'A']
test_01 sample 4: Missing/invalid argument 1 of action stack: ['stack', 'C', '<NONE>']
